# Neural Hydrology — Local-Subgraph Sweep (post-meeting batch)

Tests the professor's hypothesis: **graph signal washes out at 183-basin / 6-HUC scale; it should reappear on small, locally-coherent subgraphs.**

## What this runs

6 local subgraphs (built by a shortest-path walker on the basin distance graph) × 3 conditions × 3 seeds = **54 runs**, each 5–15 min on T4.

| Subgraph | Basins | HUCs | How built |
|---|---|---|---|
| sg_midatlantic | 16 | 02/05 | walker, seed 01594950, r=120km |
| sg_ohio | 15 | 02/05 | walker, seed 03026500 |
| sg_tennessee | 15 | 03/06 | walker, seed 03455500 |
| sg_southeast | 13 | 02/03 | walker, seed 02055100 |
| sg_northeast | 16 | 02/04 | walker, seed 01516500 |
| sg_texas_pilot | 23 | 12 | historical pilot (climate-coherent anchor) |

Conditions per subgraph:
- **L** — NH cudalstm baseline (field standard)
- **G** — DirectedGraphLSTM, empty edges (architecture-matched control)
- **G+T+M** — DirectedGraphLSTM, full edges + topology features (the full model)

## The tracked invariant

Per-seed median NSE, reported as **mean ± std across 3 seeds**. This is the quantity that should stay stable run-to-run; we watch whether `G+T+M − L` turns **positive** on the small local subgraphs (it was negative at 183-basin scale).

## How to use

1. Runtime → Change runtime type → **T4 GPU**. Save.
2. Runtime → Run all.
3. Idempotent: completed runs skip on re-run; safe to re-run after disconnects.

Outputs land in `runs/local_subgraphs/<subgraph>/<cond>_seed<N>/` on Drive.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configuration

In [ ]:
import os

# === USER CONFIG ===
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''  # leave empty for auto-detection
MODE = 'full'           # 'demo' (1 seed) or 'full' (3 seeds)
# ====================

AUTO_DETECT_CANDIDATES = [
    '/content/drive/MyDrive/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydro/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydrology/datasets/camels_us',
    '/content/drive/MyDrive/camels_us',
    '/content/drive/MyDrive/data/camels_us',
]
if not DRIVE_CAMELS_PATH:
    for cand in AUTO_DETECT_CANDIDATES:
        if os.path.isdir(cand):
            DRIVE_CAMELS_PATH = cand
            print(f'Auto-detected camels_us at: {cand}')
            break
    else:
        raise RuntimeError('Could not find camels_us. Set DRIVE_CAMELS_PATH explicitly.')
else:
    assert os.path.isdir(DRIVE_CAMELS_PATH)

topo_file = os.path.join(DRIVE_CAMELS_PATH, 'camels_attributes_v2.0', 'camels_topo.txt')
assert os.path.isfile(topo_file), f'Expected {topo_file} — does the folder have CAMELS contents?'
print('Verified camels_topo.txt present.')

DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

SEEDS = [11] if MODE == 'demo' else [11, 13, 17]

# The 6 subgraphs (built locally by build_local_subgraphs.py, checked into the repo).
SUBGRAPHS = ['sg_midatlantic', 'sg_ohio', 'sg_tennessee',
             'sg_southeast', 'sg_northeast', 'sg_texas_pilot']
# Conditions per subgraph. L must be first (graph variants use it for cfg+scaler).
CONDITIONS = ['L', 'G', 'G_T_M']
RUN_ROOT_REL = 'runs/local_subgraphs'

print(f'MODE={MODE}  SEEDS={SEEDS}')
print(f'{len(SUBGRAPHS)} subgraphs × {len(CONDITIONS)} conditions × {len(SEEDS)} seeds = '
      f'{len(SUBGRAPHS)*len(CONDITIONS)*len(SEEDS)} runs')

## Cell 3 — Clone repo

In [ ]:
REPO_DIR = '/content/nh'
import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 3

## Cell 4 — Install deps (numpy<2 pin)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith('numpy') or mod.startswith('pandas'):
        del sys.modules[mod]
import numpy as np, pandas as pd, torch
_ = torch.from_numpy(np.array([1.0])); _ = pd.DataFrame({'a':[1]})
print(f'numpy {np.__version__}  pandas {pd.__version__}  torch {torch.__version__}  CUDA {torch.cuda.is_available()}')
USE_COMPILE = int(torch.__version__.split('.')[0]) >= 2
print(f'torch.compile enabled: {USE_COMPILE}')

## Cell 5 — Symlink data + runs from Drive

In [ ]:
%cd {REPO_DIR}
REPO_DATA = os.path.join(REPO_DIR, 'datasets', 'camels_us')
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
if os.path.islink(REPO_DATA) or os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.symlink(DRIVE_CAMELS_PATH, REPO_DATA)
REPO_RUNS = os.path.join(REPO_DIR, 'runs')
if os.path.islink(REPO_RUNS) or os.path.isdir(REPO_RUNS):
    !rm -rf {REPO_RUNS}
os.symlink(DRIVE_RUNS, REPO_RUNS)
os.makedirs(os.path.join(REPO_DIR, RUN_ROOT_REL), exist_ok=True)
print(f'datasets/camels_us -> {DRIVE_CAMELS_PATH}')
print(f'runs/ -> {DRIVE_RUNS}')
!ls datasets/camels_us | head -3

## Cell 6 — GPU check

In [ ]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 7 — (Re)build the local subgraphs

Regenerates the 6 subgraph basin-lists + edges from `component0_edges.csv` via the shortest-path walker. They're already checked into the repo, so this just confirms they're present and reproducible.

In [ ]:
%cd {REPO_DIR}
!python experiments/local_subgraphs/build_local_subgraphs.py 2>&1 | tail -10

## Cell 8 — Run the sweep (6 subgraphs × 3 conditions × 3 seeds)

Per-subgraph: trains L first (graph variants need its cfg+scaler), then G, then G+T+M. Idempotent skip-if-done on `test_metrics.csv`. Cascade detection aborts loudly if the kernel wedges (the failure mode from the 5cond run).

In [ ]:
%cd {REPO_DIR}
import time, glob

compile_flag = ['--use-compile'] if USE_COMPILE else []
consecutive_fast_fails = 0

for sg in SUBGRAPHS:
    print(f'\n############ SUBGRAPH {sg} ############')
    t_sg = time.time()
    # The sweep script handles L-first ordering, skip-if-done, and baseline wiring.
    cmd = (f"python experiments/local_subgraphs/run_subgraph_sweep.py "
           f"--subgraph {sg} --seeds {' '.join(str(s) for s in SEEDS)} "
           f"--conditions {' '.join(CONDITIONS)} --device cuda:0 "
           f"{'--use-compile' if USE_COMPILE else ''}")
    !{cmd}
    elapsed = (time.time() - t_sg) / 60
    # Cascade check: a whole subgraph finishing in < 1 min means nothing trained.
    done = len(glob.glob(f'{REPO_DIR}/{RUN_ROOT_REL}/{sg}/*/test_metrics.csv')) + \
           len(glob.glob(f'{REPO_DIR}/{RUN_ROOT_REL}/{sg}/*/test/model_epoch030/test_metrics.csv'))
    print(f'  subgraph {sg}: {elapsed:.1f} min, {done} runs with metrics')
    if elapsed < 1 and done == 0:
        consecutive_fast_fails += 1
        if consecutive_fast_fails >= 2:
            raise RuntimeError('ABORT: 2 consecutive subgraphs produced no runs. '
                               'Kernel likely wedged. Restart runtime, re-run cells 1-6 + Cell 8.')
    else:
        consecutive_fast_fails = 0

## Cell 9 — The loss-distribution invariant (mean ± std across seeds)

In [ ]:
%cd {REPO_DIR}
!python experiments/local_subgraphs/analyze_subgraphs.py 2>&1 | tail -50

## Cell 10 — Show the invariant table + key contrast

The single number to watch: **G+T+M − L** per subgraph. Positive on a subgraph = the paper claim ('graph features beat standard LSTM') holds there.

In [ ]:
%cd {REPO_DIR}
import pandas as pd
from pathlib import Path
A = Path(REPO_DIR) / 'experiments' / 'local_subgraphs' / 'analysis'
inv = A / 'invariant_table.csv'
con = A / 'contrasts.csv'
if inv.exists():
    print('=== INVARIANT (median NSE, mean ± std across seeds) ===')
    df = pd.read_csv(inv)
    piv = df.pivot(index='subgraph', columns='condition', values='median_mean')
    print(piv.round(3).to_string())
if con.exists():
    print('\n=== KEY CONTRAST: G+T+M − L (paired per-basin median ΔNSE) ===')
    c = pd.read_csv(con)
    gtm = c[c['contrast'] == 'GTM_minus_L'][['subgraph', 'median', 'frac_positive']]
    print(gtm.round(3).to_string(index=False))
    print('\nPositive median = graph features beat standard LSTM on that subgraph.')
print('\nFull writeup: experiments/local_subgraphs/analysis/INVARIANT.md')

## Done

Pull `experiments/local_subgraphs/analysis/INVARIANT.md` + the CSVs locally, then in chat type **`crs interpret local subgraph results`**.

Outputs on Drive: `neural_hydrology_runs/local_subgraphs/<subgraph>/<cond>_seed<N>/`.